# My Workout Analytics (FitNotes Dataset)

1. Import libraries
2. Data overview
3. Data cleanup ?
4. Feature engineering
5. Statistics & Analytics

In [102]:
import pandas as pd
import numpy as np

This notebook begins by loading the latest FitNotes export and preparing it for analysis.

In [103]:
# Load CSV
def load_fitnotes_csv(filename=None):
    from glob import glob
    from os import path
    # Get latest csv
    dataset_filename = max(glob("FitNotes_Export_*.csv"), key=path.getmtime) if filename is None else filename
    # Return as pandas dataframe
    return pd.read_csv(dataset_filename)

df = load_fitnotes_csv()

# Overview
df.tail()

,Date,Exercise,Category,Weight,Weight Unit,Reps,Distance,Distance Unit,Time,Comment
4061,2026-06-05,Decline Sit Up,Abs,1.0,kgs,9.0,NaN,NaN,NaN,NaN
4062,2026-06-05,Barbell Curl,Biceps,17.5,kgs,12.0,NaN,NaN,NaN,NaN
4063,2026-06-05,Barbell Curl,Biceps,17.5,kgs,8.0,NaN,NaN,NaN,NaN
4064,2026-06-05,Rope Push Down,Triceps,12.5,kgs,15.0,NaN,NaN,NaN,NaN
4065,2026-06-05,Rope Push Down,Triceps,12.5,kgs,12.0,NaN,NaN,NaN,NaN


In [104]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4066 entries, 0 to 4065
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Date           4066 non-null   object 
 1   Exercise       4066 non-null   object 
 2   Category       4066 non-null   object 
 3   Weight         3899 non-null   float64
 4   Weight Unit    3899 non-null   object 
 5   Reps           3894 non-null   float64
 6   Distance       167 non-null    float64
 7   Distance Unit  167 non-null    object 
 8   Time           172 non-null    object 
 9   Comment        176 non-null    object 
dtypes: float64(3), object(7)
memory usage: 317.8+ KB


## Feature Engineering

While FitNotes provides only basic exercise categorization, we can enrich the dataset by adding our own classifications, such as primary and secondary muscle groups, movement patterns (push, pull, hinge, squat), and whether an exercise uses external load or bodyweight.

### Add Exercise metadata

First, create an Exercise catalog:

In [105]:
exercise_catalog = sorted(df["Exercise"].unique())

print(len(exercise_catalog), "total exercises:")

for exercise in exercise_catalog:
   print(exercise)

197 total exercises:
1.5 Back Squat
90° Nordic Curl
Adv. Tuck/L Front Lever Row
Alisa Squat
Alternating Bulgarian Split Squat
Alternating Single Leg Curl
Alternating Single Leg Extension
Archer Pull Up Negative
Archer Push Up
Archer Ring Push Up
Archer Ring Row
Archer Squat
BW Bar Overhead Tricep Extension
BW Bicep Curl
BW Face Pull
Back Lever Prog.
Band Assisted Muscle Up
Barbell 21s Curl
Barbell Calf Raise
Barbell Curl
Barbell Front Squat
Barbell Hack Squat
Barbell Hip Thrust
Barbell Row
Barbell Squat
Barbell Triceps Overhead Extension
Biking
Bottom Squat Hold
Bulgarian Split Squat
Cable Crunch
Cable Face Pull
Cable Hammer Curl
Cable Lateral Raise
Cable Overhead Triceps Extension
Calf Raise
Chest Press Machine
Chin Up
Clamshell
Climbing
Copenhagen Hip Dip
Copenhagen Plank
Crab Walks
Crunch
Curtsy Squat
DB Romanian Deadlift
DB Single-leg RDL
Deadlift
Decline Pike Push Up
Decline Push Up
Decline Ring Push Up
Decline Sit Up
Deficit Bulgarian Split Squat
Deficit Decline Push Up
Deficit P

In [106]:
# Now add empty metadata columns
catalog_df = pd.DataFrame({"Exercise": exercise_catalog})

catalog_df['muscles_primary'] = "" # primary muscle group exercised
catalog_df['muscles_secondary'] = "" # secondary muscle group worked
catalog_df['exercise_complexity'] = "" # compound or isolation 
catalog_df['exercise_level'] = "" # beginner, intermediate, advanced
catalog_df['movement_pattern'] = "" # push pull squat hinge, lunge; see table
catalog_df['body_region'] = "" # upper, lower, core, fullbody
catalog_df['skill_category'] = "" # strength foundation, handstand, muscle up, planche, font lever (group progressions)
catalog_df['laterality'] = "" # uni, bi
catalog_df['resistance_type'] = "" # bodyweight, free weights, machine, band
catalog_df['equipment'] = "" # bar, rings, parallel bars, floor, db, barbell, machine, cable, band

# Export
catalog_df.to_csv("exercise_metadata_empty.csv", index=False)


Open that CSV and edit manually using a spreadsheet software.

Fill it like:
| Exercise      | muscle_primary | movement_pattern | resistance_type |
| ------------- | -------------- | ---------------- | --------------- |
| Pull Up       | Lats           | Vertical Pull    | Bodyweight      |
| Dip           | Chest          | Vertical Push    | Bodyweight      |
| Barbell Squat | Quads          | Squat            | Free Weight     |

Delete any exercises you don't care about.

Save as `exercise_metadata.csv`

In [107]:
# Load it back
metadata = pd.read_csv("exercise_metadata.csv")

# Compare / diff
df_exercises = set(df["Exercise"].unique())
metadata_exercises = set(metadata["Exercise"].unique())

# In df but not in metadata
missing_in_metadata = df_exercises - metadata_exercises

# In metadata but not in df
missing_in_df = metadata_exercises - df_exercises

print("Exercises in tracker but NOT in metadata (add them or they will be dropped):")
print(sorted(missing_in_metadata))

print("\nExercises in metadata but NOT in tracker (ignore):")
print(sorted(missing_in_df))

Exercises in tracker but NOT in metadata (add them or they will be dropped):
['Biking', 'Climbing', 'Running (Outdoor)', 'Running (Treadmill)']

Exercises in metadata but NOT in tracker (ignore):
[]


If everything is OK, proceed to merge into the main dataset.

In [108]:
# Clean up metadata table
metadata = metadata[metadata["Exercise"].isin(df_exercises)]

# Merge into tracker df
df = df.merge(metadata, on="Exercise", how="right")

# Drop exercises without metadata
df = df[df["Exercise"].isin(metadata_exercises)]

# Convert Date object into datetime
df['Date'] = pd.to_datetime(df['Date'])

# Convert Time string to seconds
def convert_time_s(time_str):

    if pd.isna(time_str):
        return np.nan

    h, m, s = time_str.split(':')
    return int(h) * 3600 + int(m) * 60 + int(s)

df['Time'] = df['Time'].apply(convert_time_s)

# New overview
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4060 entries, 0 to 4059
Data columns (total 20 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Date                 4060 non-null   datetime64[ns]
 1   Exercise             4060 non-null   object        
 2   Category             4060 non-null   object        
 3   Weight               3899 non-null   float64       
 4   Weight Unit          3899 non-null   object        
 5   Reps                 3894 non-null   float64       
 6   Distance             161 non-null    float64       
 7   Distance Unit        161 non-null    object        
 8   Time                 166 non-null    float64       
 9   Comment              176 non-null    object        
 10  muscles_primary      4060 non-null   object        
 11  muscles_secondary    3018 non-null   object        
 12  exercise_complexity  4060 non-null   object        
 13  exercise_level       4060 non-nul

### Derived features

Now we can add new features based on the existing data.

In [109]:
# First get the real weight of bodyweight exercises:
BODYWEIGHT = 53

df["real_weight"] = np.where(
    # If resistance type is bodyweight, calculate real weight,
    (df["resistance_type"] == "bodyweight") & (df['Weight'].notna()),
    np.where(
        # if Weight > 1.0:
        df["Weight"] > 1.0,
        # Use BW + added weight,
        BODYWEIGHT + df["Weight"],
        # Else: use BW only.
        BODYWEIGHT
    ),
    # Else, use the set Weight
    df["Weight"]
)

df[df['resistance_type'] == "bodyweight"]

,Date,Exercise,Category,Weight,Weight Unit,Reps,Distance,Distance Unit,Time,Comment,...,muscles_secondary,exercise_complexity,exercise_level,movement_pattern,body_region,skill_category,laterality,resistance_type,equipment,real_weight
0,2025-10-24,Copenhagen Hip Dip,Legs,1.0,kgs,12.0,NaN,NaN,NaN,NaN,...,core,compound,intermediate,core,core,NaN,unilateral,bodyweight,NaN,53.0
1,2025-10-24,Copenhagen Hip Dip,Legs,1.0,kgs,12.0,NaN,NaN,NaN,NaN,...,core,compound,intermediate,core,core,NaN,unilateral,bodyweight,NaN,53.0
2,2025-10-24,Copenhagen Hip Dip,Legs,1.0,kgs,11.0,NaN,NaN,NaN,NaN,...,core,compound,intermediate,core,core,NaN,unilateral,bodyweight,NaN,53.0
3,2025-11-14,Copenhagen Hip Dip,Legs,1.0,kgs,12.0,NaN,NaN,NaN,NaN,...,core,compound,intermediate,core,core,NaN,unilateral,bodyweight,NaN,53.0
4,2025-11-14,Copenhagen Hip Dip,Legs,1.0,kgs,12.0,NaN,NaN,NaN,NaN,...,core,compound,intermediate,core,core,NaN,unilateral,bodyweight,NaN,53.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4051,2026-05-14,Wall Hand Stand,Skill,NaN,NaN,NaN,1.0,m,10.0,NaN,...,core,compound,intermediate,skill,upper,handstand,bilateral,bodyweight,NaN,NaN
4052,2026-05-14,Wall Hand Stand,Skill,NaN,NaN,NaN,1.0,m,10.0,NaN,...,core,compound,intermediate,skill,upper,handstand,bilateral,bodyweight,NaN,NaN
4053,2026-06-01,Wall Hand Stand,Skill,NaN,NaN,NaN,1.0,m,10.0,3m rest,...,core,compound,intermediate,skill,upper,handstand,bilateral,bodyweight,NaN,NaN
4054,2026-06-01,Wall Hand Stand,Skill,NaN,NaN,NaN,1.0,m,12.0,NaN,...,core,compound,intermediate,skill,upper,handstand,bilateral,bodyweight,NaN,NaN


In [110]:
# Now we can calculate the real volume of each set
df['volume'] =  df['Reps'] * df['real_weight']

In [111]:
# Add a qualitative feature - rep intensity zone:
def get_rep_intensity_zone(reps):
    if pd.isna(reps):
        return ""
    elif reps <= 5:
        return "max_strength"
    elif reps <= 12:
        return "strength_hypertrophy"
    elif reps <= 20:
        return "hypertrophy_endurance"
    else:
        return "endurance"

df['rep_zone'] = df['Reps'].apply(get_rep_intensity_zone)

df[df['Reps'].notna()]

,Date,Exercise,Category,Weight,Weight Unit,Reps,Distance,Distance Unit,Time,Comment,...,exercise_level,movement_pattern,body_region,skill_category,laterality,resistance_type,equipment,real_weight,volume,rep_zone
0,2025-10-24,Copenhagen Hip Dip,Legs,1.0,kgs,12.0,NaN,NaN,NaN,NaN,...,intermediate,core,core,NaN,unilateral,bodyweight,NaN,53.0,636.0,strength_hypertrophy
1,2025-10-24,Copenhagen Hip Dip,Legs,1.0,kgs,12.0,NaN,NaN,NaN,NaN,...,intermediate,core,core,NaN,unilateral,bodyweight,NaN,53.0,636.0,strength_hypertrophy
2,2025-10-24,Copenhagen Hip Dip,Legs,1.0,kgs,11.0,NaN,NaN,NaN,NaN,...,intermediate,core,core,NaN,unilateral,bodyweight,NaN,53.0,583.0,strength_hypertrophy
3,2025-11-14,Copenhagen Hip Dip,Legs,1.0,kgs,12.0,NaN,NaN,NaN,NaN,...,intermediate,core,core,NaN,unilateral,bodyweight,NaN,53.0,636.0,strength_hypertrophy
4,2025-11-14,Copenhagen Hip Dip,Legs,1.0,kgs,12.0,NaN,NaN,NaN,NaN,...,intermediate,core,core,NaN,unilateral,bodyweight,NaN,53.0,636.0,strength_hypertrophy
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4034,2026-04-09,Archer Ring Row,Back,1.0,kgs,4.0,NaN,NaN,NaN,NaN,...,advanced,horizontal pull,upper,NaN,unilateral,bodyweight,rings,53.0,212.0,max_strength
4056,2024-11-25,Single Dumbbell Floor Lateral Flys,Chest,5.5,kgs,12.0,NaN,NaN,NaN,NaN,...,beginner,NaN,upper,NaN,unilateral,free weight,dumbbell,5.5,66.0,strength_hypertrophy
4057,2024-11-25,Single Dumbbell Floor Lateral Flys,Chest,5.5,kgs,12.0,NaN,NaN,NaN,NaN,...,beginner,NaN,upper,NaN,unilateral,free weight,dumbbell,5.5,66.0,strength_hypertrophy
4058,2024-11-25,Single Dumbbell Floor Lateral Flys,Chest,5.5,kgs,15.0,NaN,NaN,NaN,NaN,...,beginner,NaN,upper,NaN,unilateral,free weight,dumbbell,5.5,82.5,hypertrophy_endurance


### Split into Reps x Weight and Time-based

This dataset contains two types of exercises: *reps x weight* and *distance x time*. See if they need cleaning.

In [112]:
print("# 1) REPS X WEIGHT entries:")
weight_df = df[df['Weight'].notna()]
weight_exercises = weight_df['Exercise'].unique()
print(f"{len(weight_exercises)} Exercises =", weight_exercises)

# Drop irrelevant columns
weight_df = weight_df.drop(columns=['Distance', 'Distance Unit', 'Time'])
weight_df

# 1) REPS X WEIGHT entries:
181 Exercises = ['Copenhagen Hip Dip' 'Clamshell' 'Band Assisted Muscle Up'
 'Barbell Hip Thrust' 'DB Romanian Deadlift' 'Hip Abductor Machine'
 'Hip Adductor Machine' 'Adv. Tuck/L Front Lever Row' 'Archer Push Up'
 'Archer Ring Push Up' 'BW Face Pull' 'DB Single-leg RDL' 'BW Bicep Curl'
 'Barbell 21s Curl' 'Chin Up' 'Explosive High Pull Up' 'Barbell Curl'
 'Barbell Calf Raise' 'Cable Hammer Curl' 'Deadlift' 'Glute Bridge'
 'Good Morning' 'Jump Assisted MU' 'Kettlebell Swing' 'Dumbbell Curl'
 'Calf Raise' 'Single Leg Calf Raise' 'Dragon Flag' 'L-Sit Pull Up'
 'Dumbbell Hammer Curl' 'Dumbbell Preacher Curl' 'Dumbbell Reverse Curl'
 'Smith Machine Calf Raise' 'Chest Press Machine' 'MU Negative'
 'Standing Calf Raise Machine' 'Romanian Deadlift' 'Hanging Leg Raise'
 'Single Leg Barbell RDL' 'Stiff-Legged Deadlift' 'Sumo Deadlift'
 '1.5 Back Squat' 'Alisa Squat' 'Decline Pike Push Up' 'Decline Push Up'
 'Decline Ring Push Up' 'Decline Sit Up' 'Archer Squat'
 'De

,Date,Exercise,Category,Weight,Weight Unit,Reps,Comment,muscles_primary,muscles_secondary,exercise_complexity,exercise_level,movement_pattern,body_region,skill_category,laterality,resistance_type,equipment,real_weight,volume,rep_zone
0,2025-10-24,Copenhagen Hip Dip,Legs,1.0,kgs,12.0,NaN,adductors,core,compound,intermediate,core,core,NaN,unilateral,bodyweight,NaN,53.0,636.0,strength_hypertrophy
1,2025-10-24,Copenhagen Hip Dip,Legs,1.0,kgs,12.0,NaN,adductors,core,compound,intermediate,core,core,NaN,unilateral,bodyweight,NaN,53.0,636.0,strength_hypertrophy
2,2025-10-24,Copenhagen Hip Dip,Legs,1.0,kgs,11.0,NaN,adductors,core,compound,intermediate,core,core,NaN,unilateral,bodyweight,NaN,53.0,583.0,strength_hypertrophy
3,2025-11-14,Copenhagen Hip Dip,Legs,1.0,kgs,12.0,NaN,adductors,core,compound,intermediate,core,core,NaN,unilateral,bodyweight,NaN,53.0,636.0,strength_hypertrophy
4,2025-11-14,Copenhagen Hip Dip,Legs,1.0,kgs,12.0,NaN,adductors,core,compound,intermediate,core,core,NaN,unilateral,bodyweight,NaN,53.0,636.0,strength_hypertrophy
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4034,2026-04-09,Archer Ring Row,Back,1.0,kgs,4.0,NaN,lats,biceps,compound,advanced,horizontal pull,upper,NaN,unilateral,bodyweight,rings,53.0,212.0,max_strength
4056,2024-11-25,Single Dumbbell Floor Lateral Flys,Chest,5.5,kgs,12.0,NaN,chest,NaN,isolation,beginner,NaN,upper,NaN,unilateral,free weight,dumbbell,5.5,66.0,strength_hypertrophy
4057,2024-11-25,Single Dumbbell Floor Lateral Flys,Chest,5.5,kgs,12.0,NaN,chest,NaN,isolation,beginner,NaN,upper,NaN,unilateral,free weight,dumbbell,5.5,66.0,strength_hypertrophy
4058,2024-11-25,Single Dumbbell Floor Lateral Flys,Chest,5.5,kgs,15.0,NaN,chest,NaN,isolation,beginner,NaN,upper,NaN,unilateral,free weight,dumbbell,5.5,82.5,hypertrophy_endurance


In [113]:
weight_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3899 entries, 0 to 4059
Data columns (total 20 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Date                 3899 non-null   datetime64[ns]
 1   Exercise             3899 non-null   object        
 2   Category             3899 non-null   object        
 3   Weight               3899 non-null   float64       
 4   Weight Unit          3899 non-null   object        
 5   Reps                 3894 non-null   float64       
 6   Comment              169 non-null    object        
 7   muscles_primary      3899 non-null   object        
 8   muscles_secondary    2889 non-null   object        
 9   exercise_complexity  3899 non-null   object        
 10  exercise_level       3899 non-null   object        
 11  movement_pattern     2898 non-null   object        
 12  body_region          3899 non-null   object        
 13  skill_category       517 non-null    o

In [114]:
print("# 2) DISTANCE X TIME entries:")
time_df = df[df['Time'].notna()]
time_exercises = time_df['Exercise'].unique()
print(f"{len(time_exercises)} Exercises =", time_exercises)

# Drop irrelevant columns
time_df = time_df.drop(columns=['Weight', 'Weight Unit', 'Reps', 'real_weight', 'volume', 'rep_zone'])
# These entries are actually only time based, so I'll drop Distance
if len(time_df[time_df['Distance'] > 1.0]) > 0:
    print(">>> Warning!! Non-unit Distance values found.")
time_df = time_df.drop(columns=['Distance', 'Distance Unit'])
time_df

# 2) DISTANCE X TIME entries:
13 Exercises = ['Copenhagen Plank' 'L-Sit' 'Bottom Squat Hold' 'Plank' 'Star Plank'
 'Superman Hold' 'Back Lever Prog.' 'Front Lever Prog.'
 'Hip Rotation Frog Stretch' 'Hand Stand' 'Planche Lean' 'Wall Sit'
 'Wall Hand Stand']


,Date,Exercise,Category,Time,Comment,muscles_primary,muscles_secondary,exercise_complexity,exercise_level,movement_pattern,body_region,skill_category,laterality,resistance_type,equipment
357,2025-08-08,Copenhagen Plank,Legs,20.0,NaN,adductors,core,compound,intermediate,core,core,NaN,unilateral,bodyweight,NaN
358,2025-08-08,Copenhagen Plank,Legs,20.0,NaN,adductors,core,compound,intermediate,core,core,NaN,unilateral,bodyweight,NaN
1056,2025-01-03,L-Sit,Skill,15.0,NaN,abs,hip flexors,compound,intermediate,core,core,NaN,bilateral,bodyweight,parallel bars
1057,2025-01-03,L-Sit,Skill,15.0,NaN,abs,hip flexors,compound,intermediate,core,core,NaN,bilateral,bodyweight,parallel bars
1058,2025-01-03,L-Sit,Skill,15.0,NaN,abs,hip flexors,compound,intermediate,core,core,NaN,bilateral,bodyweight,parallel bars
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4051,2026-05-14,Wall Hand Stand,Skill,10.0,NaN,shoulders,core,compound,intermediate,skill,upper,handstand,bilateral,bodyweight,NaN
4052,2026-05-14,Wall Hand Stand,Skill,10.0,NaN,shoulders,core,compound,intermediate,skill,upper,handstand,bilateral,bodyweight,NaN
4053,2026-06-01,Wall Hand Stand,Skill,10.0,3m rest,shoulders,core,compound,intermediate,skill,upper,handstand,bilateral,bodyweight,NaN
4054,2026-06-01,Wall Hand Stand,Skill,12.0,NaN,shoulders,core,compound,intermediate,skill,upper,handstand,bilateral,bodyweight,NaN


In [115]:
time_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 166 entries, 357 to 4055
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Date                 166 non-null    datetime64[ns]
 1   Exercise             166 non-null    object        
 2   Category             166 non-null    object        
 3   Time                 166 non-null    float64       
 4   Comment              7 non-null      object        
 5   muscles_primary      166 non-null    object        
 6   muscles_secondary    134 non-null    object        
 7   exercise_complexity  166 non-null    object        
 8   exercise_level       166 non-null    object        
 9   movement_pattern     151 non-null    object        
 10  body_region          166 non-null    object        
 11  skill_category       64 non-null     object        
 12  laterality           166 non-null    object        
 13  resistance_type      166 non-null    

With this added structure, we can analyze training volume over time, track muscle-group balance, measure progression in load or reps, examine consistency and frequency, and explore trends such as intensity cycles or recovery gaps. The goal is to transform simple workout logs into a structured dataset that reveals meaningful patterns in training behavior and performance.